# GoalTactics Bots Full-Audit Analysis

This notebook analyzes a **full-audit long simulation** run and focuses on:
- Team strength evolution over seasons/matchdays
- Player strength evolution and distribution drift
- Transfer and match activity
- Endpoint-level failures and bot decision patterns

It assumes the bots run writes these files:
- `api-calls.ndjson`
- `session-results.ndjson`
- `team-snapshots.ndjson`
- `squad-snapshots.ndjson`
- `errors.ndjson`

In [5]:
from pathlib import Path
import json
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 200)

SIM_ROOT = Path('/mnt/website/goal_tactics/simulations')
RUN_DIR = None  # optionally set Path('.../bots_fullaudit_...')

if RUN_DIR is None:
    patterns = [
        'bots_fullaudit_16teams_2s_*',
        'bots_fullaudit_fast16_20s_*',
        'bots_fullaudit_50s_t123_*'
    ]
    candidates = []
    for pat in patterns:
        candidates.extend(SIM_ROOT.glob(pat))
    candidates = sorted(candidates, key=lambda p: p.stat().st_mtime, reverse=True)
    if not candidates:
        raise FileNotFoundError('No bots_fullaudit run directories found for expected patterns.')
    RUN_DIR = candidates[0]

sim_dirs = sorted(RUN_DIR.glob('bots_client_sim_*'), key=lambda p: p.stat().st_mtime, reverse=True)
if not sim_dirs:
    raise FileNotFoundError(f'No bots_client_sim_* directory found under {RUN_DIR}')
SIM_DIR = sim_dirs[0]

print('RUN_DIR:', RUN_DIR)
print('SIM_DIR:', SIM_DIR)

RUN_DIR: /mnt/website/goal_tactics/simulations/bots_fullaudit_16teams_2s_20260318_071818
SIM_DIR: /mnt/website/goal_tactics/simulations/bots_fullaudit_16teams_2s_20260318_071818/bots_client_sim_20260318_061851


In [12]:
def read_ndjson(path: Path) -> pd.DataFrame:
    if not path.exists():
        print(f'Missing file: {path}')
        return pd.DataFrame()
    rows = []
    with path.open('r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                rows.append(json.loads(line))
            except json.JSONDecodeError:
                # keep notebook robust if a line is truncated
                continue
    return pd.DataFrame(rows)

api_calls = read_ndjson(SIM_DIR / 'api-calls.ndjson')
sessions = read_ndjson(SIM_DIR / 'session-results.ndjson')
team_snaps = read_ndjson(SIM_DIR / 'team-snapshots.ndjson')
squad_snaps = read_ndjson(SIM_DIR / 'squad-snapshots.ndjson')
errors = read_ndjson(SIM_DIR / 'errors.ndjson')

for df_name, df in [('api_calls', api_calls), ('sessions', sessions), ('team_snaps', team_snaps), ('squad_snaps', squad_snaps), ('errors', errors)]:
    print(df_name, df.shape)

api_calls (7784, 12)
sessions (446, 10)
team_snaps (446, 10)
squad_snaps (449, 7)
errors (0, 0)


In [7]:
# Normalize timestamps and core numeric fields
for df in [api_calls, sessions, team_snaps, squad_snaps, errors]:
    if 'tsUtc' in df.columns:
        df['tsUtc'] = pd.to_datetime(df['tsUtc'], errors='coerce', utc=True)

for col in ['season', 'matchday', 'durationMs']:
    if col in api_calls.columns:
        api_calls[col] = pd.to_numeric(api_calls[col], errors='coerce')

for col in ['season', 'matchday']:
    for df in [sessions, team_snaps, squad_snaps, errors]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')

api_calls.head(3)

,tsUtc,botId,season,matchday,phase,endpoint,success,statusCode,durationMs,requestJson,outputJson,error
0,2026-03-18 06:18:52.475205+00:00,3b367b751fdb45439b64ab33d29b4cbd,1,1,session,GetMyResources,True,200,193,null,"{""money"":10000000.0,""medipacks"":3.0,""gtStars"":...",NaN
1,2026-03-18 06:18:53.392253400+00:00,3b367b751fdb45439b64ab33d29b4cbd,1,1,session,GetMyResources,True,200,16,null,"{""money"":10000000.0,""medipacks"":3.0,""gtStars"":...",NaN
2,2026-03-18 06:18:53.461478100+00:00,3b367b751fdb45439b64ab33d29b4cbd,1,1,session,GetSquad,True,200,57,null,"{""players"":[{""id"":""72453427-7c4a-41b0-9f8d-cdf...",NaN


## 1) High-Level Run Validation

In [8]:
summary = {
    'api_calls': len(api_calls),
    'sessions': len(sessions),
    'team_snapshots': len(team_snaps),
    'squad_snapshots': len(squad_snaps),
    'error_events': len(errors),
    'bots_seen': sessions['botId'].nunique() if 'botId' in sessions.columns and not sessions.empty else 0,
    'max_season_seen': int(pd.to_numeric(sessions.get('season', pd.Series(dtype='float')), errors='coerce').max()) if not sessions.empty and 'season' in sessions.columns else None,
    'max_matchday_seen': int(pd.to_numeric(sessions.get('matchday', pd.Series(dtype='float')), errors='coerce').max()) if not sessions.empty and 'matchday' in sessions.columns else None,
}
pd.DataFrame([summary]).T.rename(columns={0: 'value'})

,value
api_calls,4716
sessions,243
team_snapshots,243
squad_snapshots,243
error_events,0
bots_seen,15
max_season_seen,1
max_matchday_seen,17


In [9]:
if not sessions.empty:
    sessions_agg = sessions.groupby(['season', 'matchday'], as_index=False).agg(
        sessions=('botId', 'count'),
        success=('success', 'sum'),
        auth_fail=('authFailed', 'sum'),
        night=('nightSession', 'sum')
    )
    sessions_agg['success_rate'] = np.where(sessions_agg['sessions'] > 0, sessions_agg['success'] / sessions_agg['sessions'], np.nan)
    display(sessions_agg.tail(20))
else:
    print('No session data yet.')

,season,matchday,sessions,success,auth_fail,night,success_rate
0,1,1,15,10,0,0,0.666667
1,1,2,15,9,0,0,0.600000
2,1,3,15,8,0,0,0.533333
3,1,4,15,8,0,0,0.533333
4,1,5,15,7,0,0,0.466667
5,1,6,15,5,0,0,0.333333
6,1,7,15,5,0,0,0.333333
7,1,8,15,5,0,0,0.333333
8,1,9,15,5,0,0,0.333333
9,1,10,15,5,0,0,0.333333


## 2) Team Strength Evolution

In [10]:
def _extract_team_strength(payload):
    if not isinstance(payload, dict):
        return np.nan
    # Support both nested {team:{strength}} and flat {strength}
    t = payload.get('team')
    if isinstance(t, dict):
        v = t.get('strength')
        if v is not None:
            return pd.to_numeric(v, errors='coerce')
    return pd.to_numeric(payload.get('strength'), errors='coerce')

if not team_snaps.empty:
    team_snaps['team_strength'] = team_snaps['team'].apply(_extract_team_strength) if 'team' in team_snaps.columns else np.nan

    team_trend = team_snaps.groupby(['season', 'matchday'], as_index=False).agg(
        avg_team_strength=('team_strength', 'mean'),
        p10=('team_strength', lambda s: s.quantile(0.10)),
        p50=('team_strength', 'median'),
        p90=('team_strength', lambda s: s.quantile(0.90)),
        teams=('botId', 'nunique')
    )
    display(team_trend.tail(30))

    # Extended team details from enriched snapshots
    if 'stadium' in team_snaps.columns:
        stadium_df = pd.json_normalize(team_snaps['stadium']).add_prefix('stadium.')
        stadium_view = pd.concat([team_snaps[['season', 'matchday', 'botId', 'teamName']].reset_index(drop=True), stadium_df], axis=1)
        display(stadium_view.tail(20))

    if 'training' in team_snaps.columns:
        training_df = pd.json_normalize(team_snaps['training']).add_prefix('training.')
        training_view = pd.concat([team_snaps[['season', 'matchday', 'botId', 'teamName']].reset_index(drop=True), training_df], axis=1)
        display(training_view.tail(20))
else:
    print('No team snapshot data yet.')

,season,matchday,avg_team_strength,p10,p50,p90,teams
0,1,1,672.666667,647.8,672.0,692.6,15
1,1,2,697.133333,664.2,695.0,733.2,15
2,1,3,697.133333,664.2,695.0,733.2,15
3,1,4,697.133333,664.2,695.0,733.2,15
4,1,5,697.133333,664.2,695.0,733.2,15
5,1,6,697.133333,664.2,695.0,733.2,15
6,1,7,697.133333,664.2,695.0,733.2,15
7,1,8,697.133333,664.2,695.0,733.2,15
8,1,9,697.133333,664.2,695.0,733.2,15
9,1,10,697.133333,664.2,695.0,733.2,15


,season,matchday,botId,teamName,stadium.buildings,stadium.maxBuildingLevel,stadium.stadium.name,stadium.stadium.capacity,stadium.stadium.grassQuality,stadium.stadium.earningsAverage
223,1,15,d308aa6df5114408bbca371c3a6a01b0,ProSilver,[{'id': 'd1709f1f-8c8e-437f-962e-3e6bf2a90dcc'...,20,My Stadium,5000,80,0
224,1,15,e1687e5b8a3e4566bbb5645a1f5fc540,EliteBolt,[{'id': 'd1709f1f-8c8e-437f-962e-3e6bf2a90dcc'...,20,My Stadium,5000,80,0
225,1,16,3b367b751fdb45439b64ab33d29b4cbd,QueenLady,[{'id': 'd1709f1f-8c8e-437f-962e-3e6bf2a90dcc'...,20,My Stadium,5000,80,0
226,1,16,40881165c3174dc8ace897fdd8446fbc,MidfielderHex,[{'id': 'd1709f1f-8c8e-437f-962e-3e6bf2a90dcc'...,20,My Stadium,5000,80,0
227,1,16,517083c87e7a45058c21a5d2995ef9e8,GoalHack,[{'id': 'd1709f1f-8c8e-437f-962e-3e6bf2a90dcc'...,20,My Stadium,5000,80,0
228,1,16,587f563873124162b08163dbe1bf8a95,LadyThunder,[{'id': 'd1709f1f-8c8e-437f-962e-3e6bf2a90dcc'...,20,My Stadium,5000,80,0
229,1,16,6c48e61f8cc545ae8e449482d987771a,ShootRain,[{'id': 'd1709f1f-8c8e-437f-962e-3e6bf2a90dcc'...,20,My Stadium,5000,80,0
230,1,16,6ced5ffb4b294a13b8a70af5ab914773,FalconDribble,[{'id': 'd1709f1f-8c8e-437f-962e-3e6bf2a90dcc'...,20,My Stadium,5000,80,0
231,1,16,7441fb35f57b4e40a8b79cfaa2085782,HeaderGamer,[{'id': 'd1709f1f-8c8e-437f-962e-3e6bf2a90dcc'...,20,My Stadium,5000,80,0
232,1,16,783570231f87486e99e7959b08719cd7,KnightDark,[{'id': 'd1709f1f-8c8e-437f-962e-3e6bf2a90dcc'...,20,My Stadium,5000,80,0


,season,matchday,botId,teamName,training.mainSkillIndex,training.subSkillIndex,training.efficiencyValue,training.noTraining
223,1,15,d308aa6df5114408bbca371c3a6a01b0,ProSilver,0,2,100,False
224,1,15,e1687e5b8a3e4566bbb5645a1f5fc540,EliteBolt,0,2,100,False
225,1,16,3b367b751fdb45439b64ab33d29b4cbd,QueenLady,0,2,100,False
226,1,16,40881165c3174dc8ace897fdd8446fbc,MidfielderHex,0,2,100,False
227,1,16,517083c87e7a45058c21a5d2995ef9e8,GoalHack,0,2,100,False
228,1,16,587f563873124162b08163dbe1bf8a95,LadyThunder,0,2,100,False
229,1,16,6c48e61f8cc545ae8e449482d987771a,ShootRain,0,2,100,False
230,1,16,6ced5ffb4b294a13b8a70af5ab914773,FalconDribble,0,2,100,False
231,1,16,7441fb35f57b4e40a8b79cfaa2085782,HeaderGamer,0,2,100,False
232,1,16,783570231f87486e99e7959b08719cd7,KnightDark,0,2,100,False


## 3) Player Strength Evolution

In [11]:
if not squad_snaps.empty and 'players' in squad_snaps.columns:
    exploded = squad_snaps[['tsUtc', 'season', 'matchday', 'botId', 'teamName', 'players']].explode('players').dropna(subset=['players']).copy()
    player_df = pd.json_normalize(exploded['players'])
    player_df.columns = [f'player.{c}' for c in player_df.columns]
    player_df = pd.concat([exploded.reset_index(drop=True), player_df.reset_index(drop=True)], axis=1)

    player_df['player.strength'] = pd.to_numeric(player_df.get('player.strength'), errors='coerce')
    player_df['player.age'] = pd.to_numeric(player_df.get('player.age'), errors='coerce')
    player_df['player.talent'] = pd.to_numeric(player_df.get('player.talent'), errors='coerce')

    # Stable player key fallback if API payload changes column naming.
    player_key = 'player.id' if 'player.id' in player_df.columns else ('player.Id' if 'player.Id' in player_df.columns else None)

    if player_key is None:
        player_df['player.synthetic_id'] = player_df.index.astype(str)
        player_key = 'player.synthetic_id'

    player_strength_trend = player_df.groupby(['season', 'matchday'], as_index=False).agg(
        avg_player_strength=('player.strength', 'mean'),
        p10=('player.strength', lambda s: s.quantile(0.10)),
        p50=('player.strength', 'median'),
        p90=('player.strength', lambda s: s.quantile(0.90)),
        players=(player_key, 'nunique')
    )
    display(player_strength_trend.tail(30))

    player_delta = player_df.sort_values([player_key, 'season', 'matchday']).copy()
    player_delta['strength_delta'] = player_delta.groupby(player_key)['player.strength'].diff()

    name_col = 'player.name' if 'player.name' in player_delta.columns else player_key
    display_cols = ['season', 'matchday', player_key, name_col, 'player.strength', 'strength_delta']
    display(player_delta[display_cols].dropna(subset=['strength_delta']).tail(30))
else:
    print('No squad snapshot data yet.')

,season,matchday,avg_player_strength,p10,p50,p90,players
0,1,1,61.585185,51.0,61.0,72.0,270
1,1,2,61.588889,51.0,61.0,72.0,270
2,1,3,61.596296,51.0,61.0,72.0,270
3,1,4,61.596296,51.0,61.0,72.0,270
4,1,5,61.600000,51.0,61.0,72.0,270
5,1,6,61.600000,51.0,61.0,72.0,270
6,1,7,61.600000,51.0,61.0,72.0,270
7,1,8,61.600000,51.0,61.0,72.0,270
8,1,9,61.600000,51.0,61.0,72.0,270
9,1,10,61.600000,51.0,61.0,72.0,270


,season,matchday,player.id,player.name,player.strength,strength_delta
425,1,2,ff4863ea-dae1-4b1e-883b-4002057ddc4d,Malte Graf,47,1.0
695,1,3,ff4863ea-dae1-4b1e-883b-4002057ddc4d,Malte Graf,47,0.0
965,1,4,ff4863ea-dae1-4b1e-883b-4002057ddc4d,Malte Graf,47,0.0
1235,1,5,ff4863ea-dae1-4b1e-883b-4002057ddc4d,Malte Graf,47,0.0
1505,1,6,ff4863ea-dae1-4b1e-883b-4002057ddc4d,Malte Graf,47,0.0
1775,1,7,ff4863ea-dae1-4b1e-883b-4002057ddc4d,Malte Graf,47,0.0
2045,1,8,ff4863ea-dae1-4b1e-883b-4002057ddc4d,Malte Graf,47,0.0
2315,1,9,ff4863ea-dae1-4b1e-883b-4002057ddc4d,Malte Graf,47,0.0
2585,1,10,ff4863ea-dae1-4b1e-883b-4002057ddc4d,Malte Graf,47,0.0
2855,1,11,ff4863ea-dae1-4b1e-883b-4002057ddc4d,Malte Graf,47,0.0


## 4) Transfers, Matches, and Endpoint Activity

In [30]:
if not api_calls.empty:
    endpoint_counts = api_calls.groupby(['endpoint', 'success'], as_index=False).size().sort_values('size', ascending=False)
    display(endpoint_counts.head(30))

    transfer_endpoints = ['SearchTransfermarket', 'BidPlayer', 'UpdateTransfermarketFavourites', 'RecruitScoutedPlayer']
    match_endpoints = ['RunMatch', 'GetLadder', 'GetLineups', 'SaveLineup']

    transfer_calls = api_calls[api_calls['endpoint'].isin(transfer_endpoints)].copy()
    match_calls = api_calls[api_calls['endpoint'].isin(match_endpoints)].copy()

    print('Transfer calls:', len(transfer_calls))
    print('Match calls:', len(match_calls))

    if not transfer_calls.empty:
        display(transfer_calls.groupby(['season', 'matchday', 'endpoint'], as_index=False).size().tail(30))

        # Transfer economics: parse bid amount and target id from request JSON.
        bid_calls = transfer_calls[transfer_calls['endpoint'] == 'BidPlayer'].copy()
        if not bid_calls.empty:
            def _parse_request_value(raw, keys):
                try:
                    payload = json.loads(raw) if isinstance(raw, str) else {}
                    for key in keys:
                        if key in payload:
                            return payload.get(key)
                    return None
                except Exception:
                    return None

            bid_calls['auction_id'] = bid_calls['requestJson'].apply(lambda s: _parse_request_value(s, ['id', 'Id']))
            bid_calls['bid_amount'] = pd.to_numeric(bid_calls['requestJson'].apply(lambda s: _parse_request_value(s, ['bid', 'Bid'])), errors='coerce')

            bid_econ = bid_calls.groupby(['season', 'matchday'], as_index=False).agg(
                bids=('endpoint', 'count'),
                successful_bids=('success', 'sum'),
                total_bid_amount=('bid_amount', 'sum'),
                avg_bid_amount=('bid_amount', 'mean')
            )
            display(bid_econ.tail(30))

            preview_cols = [c for c in ['tsUtc', 'season', 'matchday', 'botId', 'teamName', 'auction_id', 'bid_amount', 'success', 'error'] if c in bid_calls.columns]
            display(bid_calls[preview_cols].tail(30))

    if not match_calls.empty:
        display(match_calls.groupby(['season', 'matchday', 'endpoint'], as_index=False).size().tail(30))
else:
    print('No API call data yet.')

,endpoint,success,size
26,WatchAd,True,43548
7,GetMyResources,True,29598
21,SaveLineup,True,23470
15,GetSquad,True,17649
4,GetLadder,True,5483
11,GetSkillCards,True,5471
5,GetLineups,True,5471
22,SaveTeamTraining,True,5470
17,GetTeamTraining,True,5470
2,ClaimDailyReward,True,5455


Transfer calls: 3095
Match calls: 34424


,season,matchday,endpoint,size
218,8,6,SearchTransfermarket,12
219,8,7,SearchTransfermarket,12
220,8,8,SearchTransfermarket,13
221,8,9,SearchTransfermarket,12
222,8,10,SearchTransfermarket,13
223,8,11,SearchTransfermarket,13
224,8,12,SearchTransfermarket,13
225,8,13,SearchTransfermarket,13
226,8,14,SearchTransfermarket,13
227,8,15,SearchTransfermarket,13


,season,matchday,bids,successful_bids,total_bid_amount,avg_bid_amount
0,1,1,12,12,3304300,275358.333333
1,1,2,4,4,1036438,259109.500000
2,1,4,4,4,1449406,362351.500000


,tsUtc,season,matchday,botId,auction_id,bid_amount,success,error
1013,2026-03-17T06:46:05.1395392Z,1,1,7a3dbee8cd174d05bdd362cad04676c0,8d976c4d-4aa5-4be4-9da8-8e9819f530a0,254849,True,NaN
1014,2026-03-17T06:46:06.1555022Z,1,1,7a3dbee8cd174d05bdd362cad04676c0,3f636ed9-0cfe-4f87-bc25-d9acdca65d91,258633,True,NaN
1015,2026-03-17T06:46:07.1404104Z,1,1,7a3dbee8cd174d05bdd362cad04676c0,46c95ab4-72cf-4b75-a5a9-71cce336ece6,209325,True,NaN
1016,2026-03-17T06:46:08.1257539Z,1,1,7a3dbee8cd174d05bdd362cad04676c0,b4487a0f-2a88-42b7-bee9-681c101b7a2a,170106,True,NaN
1200,2026-03-17T06:46:48.2441737Z,1,1,a40df291ba3b4f2eaf01b8b84bca8936,6eb4a856-1c8f-4b4a-9390-d007f63a8deb,403230,True,NaN
1201,2026-03-17T06:46:49.245158Z,1,1,a40df291ba3b4f2eaf01b8b84bca8936,6eebfeb3-219d-4f9d-9bad-40fd97384d90,292373,True,NaN
1202,2026-03-17T06:46:50.3019971Z,1,1,a40df291ba3b4f2eaf01b8b84bca8936,8d976c4d-4aa5-4be4-9da8-8e9819f530a0,276906,True,NaN
1203,2026-03-17T06:46:51.298811Z,1,1,a40df291ba3b4f2eaf01b8b84bca8936,3f636ed9-0cfe-4f87-bc25-d9acdca65d91,279331,True,NaN
1280,2026-03-17T06:47:05.1329943Z,1,1,ac3b511cae0641e5a750dafbed5fff70,46c95ab4-72cf-4b75-a5a9-71cce336ece6,226609,True,NaN
1281,2026-03-17T06:47:06.1241495Z,1,1,ac3b511cae0641e5a750dafbed5fff70,b4487a0f-2a88-42b7-bee9-681c101b7a2a,185209,True,NaN


,season,matchday,endpoint,size
675,8,26,GetLadder,22
676,8,26,GetLineups,22
677,8,26,SaveLineup,110
678,8,27,GetLadder,22
679,8,27,GetLineups,22
680,8,27,SaveLineup,110
681,8,28,GetLadder,22
682,8,28,GetLineups,22
683,8,28,SaveLineup,110
684,8,29,GetLadder,22


## 5) Failures and Weakness Detection

In [6]:
if not api_calls.empty:
    failed_calls = api_calls[api_calls['success'] == False].copy()
    if not failed_calls.empty:
        fail_by_endpoint = failed_calls.groupby('endpoint', as_index=False).size().sort_values('size', ascending=False)
        display(fail_by_endpoint.head(25))

        failed_calls['error_text'] = failed_calls['error'].fillna('')
        failed_calls['is_429'] = failed_calls['error_text'].str.contains('429 Too Many Requests', na=False)
        failed_calls['is_422'] = failed_calls['error_text'].str.contains('422 Unprocessable Entity', na=False)
        display(failed_calls[['endpoint', 'is_429', 'is_422']].groupby('endpoint').mean().sort_values('is_429', ascending=False).head(25))
    else:
        print('No failed API calls found.')

if not errors.empty:
    display(errors.groupby(['phase'], as_index=False).size().sort_values('size', ascending=False))
    display(errors.tail(30))

,endpoint,size
2,GetMyResources,16
7,GetSquad,11
9,SaveLineup,9
3,GetMyTeamExtendedInfo,6
8,GetTeamTraining,6
11,WatchAd,5
0,ClaimDailyReward,2
1,GetLineups,2
6,GetSponsorOffers,1
5,GetSkillCards,1


,is_429,is_422
endpoint,,
ClaimDailyReward,1.0,0.0
GetLineups,1.0,0.0
GetMyResources,1.0,0.0
GetMyTeamExtendedInfo,1.0,0.0
GetScoutedPlayers,1.0,0.0
GetSkillCards,1.0,0.0
GetSponsorOffers,1.0,0.0
GetSquad,1.0,0.0
GetTeamTraining,1.0,0.0


,phase,size
0,snapshot,15


,tsUtc,botId,season,matchday,phase,message
0,2026-03-17T14:12:00.2333008Z,891c6cc41aa8418fa9ef3146ddf807ae,1,23,snapshot,Request failed: 429 Too Many Requests at /api/...
1,2026-03-17T14:17:03.5447003Z,c6a9dcc4e9574f519335a9c227b330fc,2,6,snapshot,Request failed: 429 Too Many Requests at /api/...
2,2026-03-17T14:20:06.4736478Z,891c6cc41aa8418fa9ef3146ddf807ae,2,14,snapshot,Request failed: 429 Too Many Requests at /api/...
3,2026-03-17T14:22:03.4803833Z,49954cfcc924434da7c81f59ee75eda5,2,20,snapshot,Request failed: 429 Too Many Requests at /api/...
4,2026-03-17T14:23:06.6083981Z,7c4a9ccc97344794863afc02818f4a78,2,22,snapshot,Request failed: 429 Too Many Requests at /api/...
5,2026-03-17T14:30:06.0868526Z,7c4a9ccc97344794863afc02818f4a78,3,11,snapshot,Request failed: 429 Too Many Requests at /api/...
6,2026-03-17T14:32:59.7720096Z,6089f7bd41fe444cba50afa10e9783d5,3,19,snapshot,Request failed: 429 Too Many Requests at /api/...
7,2026-03-17T14:35:00.6860866Z,693595e1abcd49148621b0e8ff8f6c61,3,24,snapshot,Request failed: 429 Too Many Requests at /api/...
8,2026-03-17T14:38:03.8679778Z,1dbdfb5564b2422191040a6ae926eb46,4,2,snapshot,Request failed: 429 Too Many Requests at /api/...
9,2026-03-17T14:47:05.0093764Z,c6a9dcc4e9574f519335a9c227b330fc,4,25,snapshot,Request failed: 429 Too Many Requests at /api/...


## 6) Export Aggregates For External Reporting

In [ ]:
EXPORT_DIR = SIM_DIR / 'analysis_exports'
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

if 'sessions_agg' in locals():
    sessions_agg.to_csv(EXPORT_DIR / 'sessions_by_matchday.csv', index=False)

if 'team_trend' in locals():
    team_trend.to_csv(EXPORT_DIR / 'team_strength_trend.csv', index=False)

if 'player_strength_trend' in locals():
    player_strength_trend.to_csv(EXPORT_DIR / 'player_strength_trend.csv', index=False)

if not api_calls.empty:
    api_calls.to_parquet(EXPORT_DIR / 'api_calls.parquet', index=False)

if not sessions.empty:
    sessions.to_parquet(EXPORT_DIR / 'session_results.parquet', index=False)

if not team_snaps.empty:
    team_snaps.to_parquet(EXPORT_DIR / 'team_snapshots.parquet', index=False)

if not squad_snaps.empty:
    squad_snaps.to_parquet(EXPORT_DIR / 'squad_snapshots.parquet', index=False)

print('Exports written to:', EXPORT_DIR)